In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import os

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
# Define transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [6]:
# Load datasets
data_dir = "D:\Ramana\Carrer\Projects\CV Gun Game\GestureDataset\HandNavigation"  # Change this to your dataset path
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")
val_dir = os.path.join(data_dir, "validation")


In [7]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [9]:
# Define CNN model
class HandGestureCNN(nn.Module):
    def __init__(self, num_classes=9):
        super(HandGestureCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(128 * 16 * 16, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [10]:
# Initialize model
model = HandGestureCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
# Training loop
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

In [12]:
# Evaluate model
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [13]:
# Train the model
train_model(model, train_loader, criterion, optimizer, epochs=10)

Epoch 1, Loss: 0.4402
Epoch 2, Loss: 0.0230
Epoch 3, Loss: 0.0183
Epoch 4, Loss: 0.0124
Epoch 5, Loss: 0.0039
Epoch 6, Loss: 0.0095
Epoch 7, Loss: 0.0101
Epoch 8, Loss: 0.0029
Epoch 9, Loss: 0.0061
Epoch 10, Loss: 0.0066


In [14]:
# Test accuracy
test_accuracy = evaluate_model(model, test_loader)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 94.22%


In [15]:
# Validation accuracy
val_accuracy = evaluate_model(model, val_loader)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

Validation Accuracy: 96.56%


In [16]:
# Save the model
torch.save(model.state_dict(), "hand_gesture_cnn.pth")
print("Model saved successfully!")

Model saved successfully!


In [17]:
import torch
import torchvision.transforms as transforms
from PIL import Image

In [20]:
# Define class labels (should match your dataset folder names)
class_labels = ["anticlockwise", "backward", "clockwise", "down", "forward", "left", "right", "up", "wave"]

# Function to predict the gesture
def predict_gesture(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)  # Add batch dimension

    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)

    return class_labels[predicted.item()]

# Example usage
image_path = r"D:\Ramana\Carrer\Projects\CV Gun Game\frame_1714026510559.png"  # Change this to your image path
prediction = predict_gesture(image_path)
print(f"Predicted Gesture: {prediction}")


Predicted Gesture: right


In [ ]:
# import tensorflow as tf
# import numpy as np
# import os

# # Data generators for loading images
# train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
# test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
# val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

# # Dataset paths
# dataset_path = "path_to_dataset"
# train_path = os.path.join(dataset_path, "train")
# test_path = os.path.join(dataset_path, "test")
# val_path = os.path.join(dataset_path, "validation")

# # Load data
# train_data = train_datagen.flow_from_directory(train_path, target_size=(128, 128), batch_size=32, class_mode='categorical')
# test_data = test_datagen.flow_from_directory(test_path, target_size=(128, 128), batch_size=32, class_mode='categorical')
# val_data = val_datagen.flow_from_directory(val_path, target_size=(128, 128), batch_size=32, class_mode='categorical')

# # Define CNN model
# model = tf.keras.Sequential([
#     tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
#     tf.keras.layers.MaxPooling2D(2,2),
#     tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),
#     tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
#     tf.keras.layers.MaxPooling2D(2,2),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dropout(0.5),
#     tf.keras.layers.Dense(len(train_data.class_indices), activation='softmax')
# ])

# # Compile the model
# model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# # Train the model
# history = model.fit(train_data, validation_data=val_data, epochs=20)

# # Save the model
# model.save("hand_gesture_cnn.h5")

# # Evaluate the model
# test_loss, test_acc = model.evaluate(test_data)
# print(f"Test Accuracy: {test_acc * 100:.2f}%")

# val_loss, val_acc = model.evaluate(val_data)
# print(f"Validation Accuracy: {val_acc * 100:.2f}%")
